In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


from torchvision import datasets,transforms
from torch.utils.data import Dataset,DataLoader


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data , batch_size=64 , shuffle=True)
test_loader = DataLoader(test_data , batch_size=64 , shuffle=False)




Select Device (GPU or CPU)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device =", device)


Define the MLP Model for MNIST Classification

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),                 # تبدیل 28x28 به 784
            nn.Linear(784, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)            # خروجی 10 کلاس
        )

    def forward(self, x):
        return self.model(x)

model = MLP().to(device)


Define Loss Function and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

Training Loop for the MLP Model

In [ ]:
loss_history = []
epochs = 5
for epoch in range(epochs):
    total_loss = 0
    for images,labels in train_loader:
        images,labels= images.to(device),labels.to(device)

        outputs=model(images)
        loss = criterion(outputs,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss+=loss.item()
    loss_history.append(total_loss)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")



Evaluate Model Accuracy on Test Dataset

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for images,labels in test_loader:
        images,labels=images.to(device),labels.to(device)
        outputs=model(images)
        _, predicted = torch.max(outputs.data, 1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
print("Accuracy =",correct/total)

Visualize Sample Predictions on MNIST Test Images

In [ ]:
import matplotlib.pyplot as plt

# گرفتن یک batch از تست
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

# پیش‌بینی مدل
outputs = model(images)
_, predicted = torch.max(outputs, 1)

# نمایش 10 تصویر اول
for i in range(3):
    img = images[i].cpu().squeeze().numpy()
    plt.imshow(img, cmap='gray')
    plt.title(f"Real: {labels[i].item()}  |  Predicted: {predicted[i].item()}")
    plt.show()


Find and Visualize Misclassified MNIST Samples

In [ ]:
wrong_images = []
wrong_labels = []
wrong_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        # پیدا کردن موارد اشتباه
        wrong = (predicted != labels)

        for i in range(len(images)):
            if wrong[i]:
                wrong_images.append(images[i].cpu())
                wrong_labels.append(labels[i].item())
                wrong_predictions.append(predicted[i].item())


for i in range(3):
    img = wrong_images[i].squeeze().numpy()
    plt.imshow(img, cmap='gray')
    plt.title(f"Real: {wrong_labels[i]} | Predicted: {wrong_predictions[i]}")
    plt.show()



Confusion Matrix for MNIST Classification

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Real")
plt.show()


Plot Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt

plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()
